# Habits-to-Grades: Exploratory Data Analysis
**DS-211 Theory of Data Science · GIK Institute**

This notebook explores the GIKI student survey dataset collected for the Habits-to-Grades project.  
We cover:
1. Dataset overview and cleaning
2. Target variable (CGPA) distribution
3. Feature distributions
4. Correlation analysis
5. Key findings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import warnings
import sys, os
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join('..', 'models'))
from train_models import load_clean_data

# ── Dark theme consistent with the app
BG, SURFACE, BORDER = '#080C10', '#0D1117', '#21262D'
CYAN, GREEN, RED, ORANGE, MUTED, TEXT = '#00E5FF', '#39D353', '#EF4444', '#F97316', '#7D8590', '#E6EDF3'

plt.rcParams.update({
    'font.family': 'monospace',
    'axes.facecolor': SURFACE,
    'figure.facecolor': BG,
    'axes.edgecolor': BORDER,
    'axes.labelcolor': MUTED,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'text.color': TEXT,
    'grid.color': BORDER,
    'grid.linestyle': '--',
    'grid.alpha': 0.4,
})

print('Libraries loaded.')

## 1. Dataset Overview

In [ ]:
# Load raw data
df_raw = pd.read_excel('../data.xlsx')
print(f'Raw dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')

# Load cleaned data
df = load_clean_data('../data.xlsx')
print(f'After cleaning: {df.shape[0]} rows retained')
print(f'Dropped: {df_raw.shape[0] - df.shape[0]} rows with missing values')
df.head(3)

In [ ]:
# Missing value audit
miss = df_raw.isnull().sum()
miss = miss[miss > 0].rename('missing')
print('Missing values per column:')
print(miss.to_string())

In [ ]:
# Gender breakdown
gender_col = df_raw.columns[3]
gender_counts = df_raw[gender_col].value_counts()
print('Gender distribution:')
print(gender_counts.to_string())

## 2. Target Variable: CGPA Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

# Histogram
ax = axes[0]
ax.hist(df['cgpa'], bins=30, color=CYAN, edgecolor=BG, alpha=0.85)
ax.axvline(df['cgpa'].mean(), color=ORANGE, linewidth=1.5, linestyle='--', label=f'Mean: {df["cgpa"].mean():.2f}')
ax.axvline(df['cgpa'].median(), color=GREEN, linewidth=1.5, linestyle='--', label=f'Median: {df["cgpa"].median():.2f}')
ax.set_title('CGPA Distribution', fontsize=12, fontweight='bold', color=TEXT)
ax.set_xlabel('CGPA', color=MUTED)
ax.set_ylabel('Count', color=MUTED)
ax.legend(fontsize=9, labelcolor=TEXT, facecolor=SURFACE, edgecolor=BORDER)
ax.yaxis.grid(True)
ax.set_axisbelow(True)

# Box plot by gender
ax = axes[1]
gender_labels = {0: 'Male', 1: 'Female', 2: 'Other'}
df['gender_label'] = df['gender_enc'].map(gender_labels)
groups = [df[df['gender_label'] == g]['cgpa'].values for g in ['Male', 'Female', 'Other']]
bp = ax.boxplot(groups, labels=['Male', 'Female', 'Other'],
                patch_artist=True, medianprops=dict(color=CYAN, linewidth=2))
colors_box = [CYAN, ORANGE, MUTED]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.3)
ax.set_title('CGPA by Gender', fontsize=12, fontweight='bold', color=TEXT)
ax.set_ylabel('CGPA', color=MUTED)
ax.yaxis.grid(True)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('../static/plots/eda_cgpa.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print(df['cgpa'].describe().round(3).to_string())

## 3. Feature Distributions

In [ ]:
num_features = [
    ('study_hours',    'Study Hours / Day'),
    ('attendance_pct', 'Attendance %'),
    ('social_media',   'Social Media Hours / Day'),
    ('sleep_hours',    'Sleep Hours / Night'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor(BG)
axes = axes.flatten()

for ax, (col, label) in zip(axes, num_features):
    ax.hist(df[col].dropna(), bins=25, color=CYAN, edgecolor=BG, alpha=0.8)
    ax.set_title(label, fontsize=11, fontweight='bold', color=TEXT)
    ax.set_xlabel(label, color=MUTED, fontsize=9)
    ax.set_ylabel('Count', color=MUTED, fontsize=9)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    mean_val = df[col].mean()
    ax.axvline(mean_val, color=ORANGE, linewidth=1.5, linestyle='--', label=f'Mean: {mean_val:.1f}')
    ax.legend(fontsize=8, labelcolor=TEXT, facecolor=SURFACE, edgecolor=BORDER)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', color=TEXT, y=1.01)
plt.tight_layout()
plt.savefig('../static/plots/eda_features.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Categorical features: diet and exercise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

# Diet quality
ax = axes[0]
diet_map_rev = {0:'Very Poor', 1:'Poor', 2:'Average', 3:'Good', 4:'Excellent'}
diet_counts = df['diet_enc'].value_counts().sort_index()
labels = [diet_map_rev[i] for i in diet_counts.index]
bar_colors = [RED, ORANGE, MUTED, CYAN, GREEN]
ax.bar(labels, diet_counts.values, color=bar_colors[:len(labels)], edgecolor=BG, alpha=0.85)
ax.set_title('Diet Quality Distribution', fontsize=11, fontweight='bold', color=TEXT)
ax.set_ylabel('Count', color=MUTED)
ax.tick_params(axis='x', rotation=15)
ax.yaxis.grid(True)
ax.set_axisbelow(True)

# Exercise days
ax = axes[1]
exercise_map_rev = {0:'0 days', 1.5:'1-2 days', 3.5:'3-4 days', 5.5:'5-6 days', 7:'7 days'}
ex_counts = df['exercise_num'].value_counts().sort_index()
ex_labels = [exercise_map_rev.get(v, str(v)) for v in ex_counts.index]
ax.bar(ex_labels, ex_counts.values, color=CYAN, edgecolor=BG, alpha=0.8)
ax.set_title('Exercise Frequency Distribution', fontsize=11, fontweight='bold', color=TEXT)
ax.set_ylabel('Count', color=MUTED)
ax.yaxis.grid(True)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('../static/plots/eda_categorical.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 4. Correlation Analysis

In [ ]:
features = ['study_hours','attendance_pct','social_media','sleep_hours',
            'exercise_num','society_count','diet_enc','age','cgpa']
labels   = ['Study Hrs','Attendance','Social Media','Sleep Hrs',
            'Exercise','Societies','Diet','Age','CGPA']

corr = df[features].rename(columns=dict(zip(features, labels))).corr()

# Print CGPA correlations ranked
cgpa_corr = corr['CGPA'].drop('CGPA').sort_values(key=abs, ascending=False)
print('Correlation with CGPA (ranked by magnitude):')
for feat, val in cgpa_corr.items():
    bar = '█' * int(abs(val) * 20)
    direction = '+' if val > 0 else '-'
    print(f'  {feat:<14} {direction}{bar:<20} {val:+.3f}')

In [ ]:
# Scatter: top 2 positively correlated features vs CGPA
top2 = cgpa_corr[cgpa_corr > 0].head(2).index.tolist()
feat_cols = [labels.index(f) for f in top2]
raw_cols  = [features[i] for i in feat_cols]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

for ax, raw_col, label in zip(axes, raw_cols, top2):
    sc = ax.scatter(df[raw_col], df['cgpa'], alpha=0.4, s=18,
                    c=df['cgpa'], cmap='cool', edgecolors='none')
    # Trend line
    m, b = np.polyfit(df[raw_col].dropna(), df.loc[df[raw_col].notna(), 'cgpa'], 1)
    x_line = np.linspace(df[raw_col].min(), df[raw_col].max(), 100)
    ax.plot(x_line, m * x_line + b, color=ORANGE, linewidth=2, linestyle='--', alpha=0.9)
    r = df[[raw_col, 'cgpa']].corr().iloc[0, 1]
    ax.set_title(f'{label} vs CGPA  (r = {r:.3f})', fontsize=11, fontweight='bold', color=TEXT)
    ax.set_xlabel(label, color=MUTED)
    ax.set_ylabel('CGPA', color=MUTED)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

plt.suptitle('Top Positive Predictors of CGPA', fontsize=13, fontweight='bold', color=TEXT, y=1.01)
plt.tight_layout()
plt.savefig('../static/plots/eda_scatter.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 5. Key Findings

From the exploratory analysis:

- **Attendance** is the strongest single predictor of CGPA in this dataset.
- **Study hours** shows a clear positive linear trend with CGPA.
- **Social media usage** is negatively correlated with CGPA — students spending more time on social media tend to have lower GPAs.
- The CGPA distribution is roughly bell-shaped with a mean around **2.83**, slightly left-skewed.
- **Diet quality** and **exercise** show weaker but directionally positive correlations with CGPA.
- **Age** has near-zero correlation — year of study is not captured, which limits its predictive power.
- The dataset is male-dominated (521 / 653 = 80%), which may limit generalizability of gender-based insights.